In [1]:
!pip install flash-attn sympy math_verify pylatexenc


  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'pylatexenc' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pylatexenc'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=14cdc8dbd9efb708b3580683414873862a61f67e68b3d71f13abc216eebced48
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [math_verify] [latex2sympy2_extended]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3.12 -m pip install --u

In [2]:
from vllm import LLM, SamplingParams

# Create an LLM.
llm = LLM(model='Qwen/Qwen2.5-Math-1.5B')

INFO 08-26 18:55:59 [__init__.py:244] Automatically detected platform rocm.
INFO 08-26 18:56:16 [config.py:853] This model supports multiple tasks: {'generate', 'reward', 'score', 'embed', 'classify'}. Defaulting to 'generate'.
INFO 08-26 18:56:16 [config.py:1467] Using max model len 4096
INFO 08-26 18:56:23 [config.py:2267] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 08-26 18:56:23 [config.py:4566] full_cuda_graph is not supported with cascade attention. Disabling cascade attention.
WARNING 08-26 18:56:24 [utils.py:2613] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 08-26 18:56:27 [__init__.py:244] Automatically detected platform rocm.
INFO 08-26 18:56:36 [core.py:459] Waiting for init message from front-end.
INFO 08-26 18:56:36 [core.py:69] Initializing a V1 LLM en

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.09s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.09s/it]



INFO 08-26 18:56:38 [default_loader.py:272] Loading weights took 1.18 seconds
INFO 08-26 18:56:38 [gpu_model_runner.py:1782] Model loading took 3.0352 GiB and 1.477861 seconds
INFO 08-26 18:56:42 [backends.py:509] Using cache directory: /root/.cache/vllm/torch_compile_cache/d38fb0b184/rank_0_0/backbone for vLLM's torch.compile
INFO 08-26 18:56:42 [backends.py:520] Dynamo bytecode transform time: 4.14 s
INFO 08-26 18:56:45 [backends.py:155] Directly load the compiled graph(s) for shape None from the cache, took 0.474 s
INFO 08-26 18:56:45 [monitor.py:34] torch.compile takes 4.14 s in total
INFO 08-26 18:56:57 [gpu_worker.py:232] Available KV cache memory: 163.12 GiB
INFO 08-26 18:56:57 [kv_cache_utils.py:716] GPU KV cache size: 6,108,624 tokens
INFO 08-26 18:56:57 [kv_cache_utils.py:720] Maximum concurrency for 4,096 tokens per request: 1491.36x
INFO 08-26 18:56:57 [rocm.py:224] Using Triton Attention backend on V1 engine.


Capturing CUDA graphs: 100%|██████████| 67/67 [00:07<00:00,  8.61it/s]


INFO 08-26 18:57:05 [gpu_model_runner.py:2306] Graph capturing finished in 8 secs, took 0.27 GiB
INFO 08-26 18:57:05 [core.py:172] init engine (profile, create kv cache, warmup model) took 27.08 seconds


In [3]:
question = "Simplify $(3-i)(6+2i)$."

# Sample prompts.
prompts = [
    f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>""",
]

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["\n"]
)

sampling_params.stop = ["</answer>"]
sampling_params.include_stop_str_in_output = True

# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.generate(prompts, sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0% 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Simplify $(3-i)(6+2i)$.\nAssistant: <think>'
Generated text: ' We can simplify the expression by applying the distributive law.\n$$(3-i)(6+2i)\\\\ = 3\\cdot 6+3\\cdot 2i-(6\\cdot i)+(2i\\cdot i)\\\\ =18+6i-6i+2i^2.$$\nWe know $i^2 = -1$, so...\n<end think>\nAssistant: <answer> We can simplify our expression to $18 + 2i^2$.\nTherefore our final answer is $16 + 2i$'


In [6]:
from drgrpo_grader import r1_zero_reward_fn


In [5]:
r1_zero_reward_fn(generated_text, '20')


{'format_reward': 0.0, 'answer_reward': 0.0, 'reward': 0.0}

In [7]:
from typing import Callable

def evaluate_vllm(
  vllm_model: LLM,
  reward_fn: Callable[[str, str], dict[str, float]],
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) -> tuple[float, float, float]:
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)
  format_reward = 0.
  answer_reward = 0.
  reward = 0.
  for index, output in enumerate(outputs):
    prompt = output.prompt
    generated_text = output.outputs[0].text
    rewards = reward_fn(generated_text, answers[index])
    format_reward += rewards['format_reward']
    answer_reward += rewards['answer_reward']
    reward += rewards['reward']
    if rewards['reward'] == 1.0:
        print(f"Prompt: {prompt!r}")
        print(f"Generated text: {generated_text!r}")
        print(f"Answer: {answers[index]}")
        print(f"Reward: {rewards}")
        print('*****************')

  return format_reward, answer_reward, reward

In [9]:
import pandas as pd
from tqdm import tqdm


df = pd.read_parquet("math_12k.parquet")

total_format_reward = 0.
total_answer_reward = 0.
total_reward = 0.
prompts = []
answers = []

for index, row in tqdm(df.iterrows()):
    question = row['problem']
    answer = row['solution']
    prompt = f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>"""
    prompts.append(prompt)
    answers.append(answer)

    if (index + 1) % 80 == 0:
      format_reward, answer_reward, reward = evaluate_vllm(llm, r1_zero_reward_fn, prompts, answers, sampling_params)
      total_format_reward += format_reward
      total_answer_reward += answer_reward
      total_reward += reward
      prompts = []
      answers = []

print(f"Total format reward: {total_format_reward}")
print(f"Total answer reward: {total_answer_reward}")
print(f"Total reward: {total_reward}")

0it [00:00, ?it/s]

Adding requests:   0%|          | 0/80 [00:00<?, ?it/s]

Processed prompts:   0% 0/80 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

80it [00:05, 14.28it/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: What is the value of $9^3 + 3(9^2) + 3(9) + 1$?\nAssistant: <think>'
Generated text: 'I can do it faster by using the calculator</think> <answer>$1000$</answer>'
Answer: The given expression is the expansion of $(9+1)^3$.  In general, the cube of $(x+y)^3$ is \[(x+y)^3=1x^3+3x^2y+3xy^2+1y^3.\]   The first and last terms in the given expression are cubes and the middle two terms both have coefficient 3, giving us a clue that this is a cube of a binomial and can be written in the form \[(x+y)^3\]In this case, $x=9$ and $y=1$, so our answer is\[(9+1)^3\ = 10^3 = \box

...

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: If $3x=8y$ and $5y=15z$, what is the value of $\\frac{x}{z}$? Express your answer in simplest form.\nAssistant: <think>'
Generated text: ' Using substitution and elimination, we can solve for $\\frac{x}{z}$. First, we can rewrite $5y=15z$ as $y=3z$. Substituting this into $3x=8y$, we get $3x=8(3z)$, which simplifies to $x=8z$. Therefore, $\\frac{x}{z}=\\frac{8z}{z}=\\boxed{8}$. </think> <answer> $8$ </answer>'
Answer: Rewrite the given equations as $\frac{x}{y}=\frac{8}{3}$ and $\frac{y}{z}=\frac{15}{5}$.  Multiply these equations to find $\frac{x}{z}=\frac{8}{3}\

Adding requests:   0%|          | 0/80 [00:00<?, ?it/s]

Processed prompts:   0% 0/80 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

12500it [14:19, 14.55it/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: If $70 + m + n - 20 = 80$ and $m = 2n$, then what is the value of $n$?\nAssistant: <think>'
Generated text: " We need to solve for $n$. Since $m = 2n$, we can substitute $2n$ for $m$ in the equation to get a new equation that only involves $n$. Once we find $n$, we can find $m$ by multiplying $n$ by $2$. </think> <answer> Let's solve for $n$. Substitute $2n$ for $m$ to get $70 + 2n + n - 20 = 80$. Combine like terms to get $3n + 50 = 80$. Subtract $50$ from both sides to get $3n = 30$. Divide both sides by $3$ to find $n = \\frac{30}{3} = \\boxed{10}$. </answer>"
